# Distillation in Transformers

This notebook explains **knowledge distillation** in transformer models: what it means, why it matters, its main types, and the step-by-step process with examples.


## 1. What is distillation?

**Knowledge distillation** is a training method where a **large, powerful model** (the **teacher**) teaches a **smaller model** (the **student**) to behave similarly.

The student tries to imitate not only the final answer, but also the teacher’s *confidence* and sometimes its internal representations.

### Main goal
- keep most of the teacher’s performance
- reduce model size
- make inference faster
- lower memory and deployment cost


## 2. Why is distillation important?

Large transformers are often too expensive for many devices and applications.

Distillation helps when you need:
- faster inference
- smaller memory usage
- cheaper cloud deployment
- models that can run on mobile, edge, or low-resource hardware

It is useful in:
- chatbots
- text classification
- speech models
- search and retrieval
- production systems where speed matters


## 3. Teacher vs student

| Model | Role | Typical size |
|---|---|---|
| Teacher | Trained large model with strong accuracy | bigger |
| Student | Smaller model trained to imitate teacher | smaller |

Example:
- Teacher: BERT-large
- Student: DistilBERT

The student is not trained from scratch in a random way only from hard labels; it also learns from the teacher’s softer output distribution.


## 4. Core idea with intuition

Suppose the teacher sees the sentence:

> "This movie is amazing"

For sentiment classification, the teacher may produce probabilities like:

| Class | Probability |
|---|---:|
| Positive | 0.92 |
| Neutral | 0.06 |
| Negative | 0.02 |

A hard label would only say: **Positive**.

The soft probabilities give more information:
- Positive is the best answer
- Neutral is somewhat possible
- Negative is very unlikely

That extra information is what the student learns from.


## 5. Why soft targets are useful

Hard labels are one-hot vectors.

Example for 3 classes:

- Positive → `[1, 0, 0]`
- Neutral → `[0, 1, 0]`
- Negative → `[0, 0, 1]`

This tells the model only the correct class.

Soft targets show class relationships. For example:

- Positive: 0.70
- Neutral: 0.20
- Negative: 0.10

Now the student sees which wrong answers are *closer* to the right answer.


## 6. Types of distillation

There is more than one way to distill knowledge.

### 6.1 Logit distillation
The student learns from the teacher’s output probabilities (logits after softmax).

This is the most common form.

### 6.2 Hidden-state distillation
The student tries to match intermediate layer representations from the teacher.

This helps the student learn internal semantic structure.

### 6.3 Attention distillation
The student imitates the teacher’s attention maps.

This can teach the student *where to focus* in the sequence.

### 6.4 Feature distillation
The student matches selected feature vectors from the teacher, often from one or more layers.

### 6.5 Sequence-level distillation
The teacher generates target sequences, and the student is trained on those generated outputs.

This is common in translation and generation tasks.

### 6.6 Self-distillation
The same architecture, or an earlier version of the same model, teaches a later version.

This can improve robustness and calibration.


## 7. How the distillation process works

A typical workflow is:

1. **Train or choose a strong teacher model**
2. **Run the teacher on training inputs**
3. **Collect soft outputs** and sometimes hidden states/attention maps
4. **Train the student model** to match the teacher
5. **Combine distillation loss with task loss**
6. **Evaluate the student** on validation/test data

The student usually has:
- fewer layers
- smaller hidden size
- fewer attention heads
- fewer parameters

So it is cheaper to run after training.


## 8. Loss functions used in distillation

A student is usually trained with two losses:

### 8.1 Task loss
This compares the student prediction with the real label.

Example:
- cross-entropy with the true class

### 8.2 Distillation loss
This compares the student prediction with the teacher prediction.

A common choice is **KL divergence**.

### Combined loss

\[
\mathcal{L} = \alpha \cdot \mathcal{L}_{task} + (1-\alpha) \cdot \mathcal{L}_{distill}
\]

Where:
- \(\alpha\) controls the balance
- task loss comes from the true labels
- distillation loss comes from the teacher

Sometimes a **temperature** term is used to soften probabilities.


## 9. Temperature in distillation

Softmax can be made smoother by dividing logits by a temperature \(T\):

\[
\text{Softmax}(z_i) = \frac{e^{z_i/T}}{\sum_j e^{z_j/T}}
\]

When \(T > 1\), the output distribution becomes softer.

This is useful because the student can learn more than just the top answer. It learns the relative similarity among classes or tokens.

Example:
- with low temperature, one class may dominate completely
- with higher temperature, the model reveals more structure in its beliefs


## 10. Example: sentiment classification

Suppose the input is:

> "The laptop is good, but the battery is weak"

The teacher might output:

| Class | Probability |
|---|---:|
| Positive | 0.45 |
| Neutral | 0.40 |
| Negative | 0.15 |

This is more informative than only saying the label is "Positive".

The student learns that this is a mixed or uncertain sentence, not a strongly positive one.


## 11. Example: language modeling

For a GPT-style model, the teacher predicts the next token.

Input:

> "The cat sat on the"

Teacher output might look like:

| Token | Probability |
|---|---:|
| mat | 0.61 |
| floor | 0.16 |
| chair | 0.08 |
| bed | 0.05 |

The student is trained to imitate this distribution, not just the top token `mat`.

That helps preserve richer language knowledge while using a smaller model.


## 12. Example: why the student can outperform a plain small model

Imagine two small models:

- **Small Model A**: trained only on hard labels
- **Small Model B**: trained with distillation from a teacher

Even if both are small, Model B often performs better because it gets extra guidance from the teacher’s soft outputs and internal patterns.

That is why distillation is powerful: it transfers useful structure, not just answers.


In [ ]:
import math
from typing import List


def softmax(logits: List[float], temperature: float = 1.0) -> List[float]:
    scaled = [x / temperature for x in logits]
    m = max(scaled)
    exps = [math.exp(x - m) for x in scaled]
    s = sum(exps)
    return [e / s for e in exps]


def kl_divergence(p: List[float], q: List[float]) -> float:
    eps = 1e-12
    return sum(pi * math.log((pi + eps) / (qi + eps)) for pi, qi in zip(p, q))


# Teacher and student logits for 3 classes: Positive, Neutral, Negative
teacher_logits = [4.2, 1.1, 0.2]
student_logits = [2.5, 0.8, 0.5]

teacher_probs_T1 = softmax(teacher_logits, temperature=1.0)
teacher_probs_T4 = softmax(teacher_logits, temperature=4.0)
student_probs_T4 = softmax(student_logits, temperature=4.0)

print('Teacher probs (T=1):', [round(p, 4) for p in teacher_probs_T1])
print('Teacher probs (T=4):', [round(p, 4) for p in teacher_probs_T4])
print('Student probs (T=4):', [round(p, 4) for p in student_probs_T4])
print('KL(teacher || student) at T=4:', round(kl_divergence(teacher_probs_T4, student_probs_T4), 6))


## 13. Distillation in transformer models specifically

In transformers, distillation can be applied to:

- **output probabilities**
- **hidden layers**
- **attention matrices**
- **sequence outputs**

Because transformers have many layers and attention heads, distillation can compress a lot of information into a smaller model.

A famous example is **DistilBERT**, which is a compressed version of BERT designed to be faster and lighter while keeping much of the accuracy.


## 14. Advantages of distillation

- smaller model size
- faster inference
- lower memory usage
- cheaper deployment
- better than training a small model from scratch in many cases
- useful for edge devices and real-time systems


## 15. Limitations of distillation

- the student usually cannot fully match the teacher
- training can be more complex than ordinary supervised learning
- poor teacher quality gives poor student quality
- some kinds of knowledge are hard to compress
- the student architecture may limit how much information can be transferred


## 16. Simple summary

Distillation is a compression strategy where a strong teacher model transfers knowledge to a smaller student model.

The student learns:
- the final answers
- the teacher’s confidence
- sometimes the teacher’s hidden representations and attention patterns

This makes the student smaller, faster, and practical for deployment.
